In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType, IntegerType, StringType, DateType
import uuid
import random
from datetime import datetime, timedelta

In [0]:
@udf(returnType=StringType())
def get_generate_transaction_id():
    """

    :return:
    """
    return str(uuid.uuid4())

@udf(returnType=IntegerType())
def get_product_id():
    """

    :return:
    """
    return random.choice(range(1, 6))

@udf(returnType=StringType())
def get_region():
    return random.choice(['Europe', 'Asia', 'North America', 'South America'])

@udf(returnType=DateType())
def get_random_date():
    start_dt = datetime.strptime("2020-01-01", "%Y-%m-%d")
    end_dt = datetime.strptime("2024-12-31", "%Y-%m-%d")
    delta = (end_dt - start_dt).days
    return (start_dt + timedelta(days=random.randint(0, delta))).date()

@udf(returnType=IntegerType())
def get_quantity_by_product_id(product_id: int | str):
    max_quantity_dict = {1: 20, 2: 20, 3: 20, 4: 20, 5: 20}
    return random.randint(1, max_quantity_dict.get(int(product_id), 1))

@udf(returnType=FloatType())
def get_price_by_product_id(product_id: int | str):
    d = {1: {'min_price': 200, 'max_price': 3000}, 2: {'min_price': 350, 'max_price': 4000},
                                    3: {'min_price': 15, 'max_price': 500}, 4: {'min_price': 30, 'max_price': 1200},
                                    5: {'min_price': 10, 'max_price': 120}}
    price_min_max = d.get(int(product_id), {})

    return round(random.uniform(price_min_max.get('min_price', 0), price_min_max.get('max_price', 0)), 2)


In [0]:
df = spark.range(1000000).withColumn('transaction_id', get_generate_transaction_id()) \
    .withColumn('product_id', get_product_id()) \
    .withColumn('region', get_region()) \
    .withColumn('transaction_date', get_random_date())

df = df.withColumn('quantity', get_quantity_by_product_id('product_id')) \
    .withColumn('price', get_price_by_product_id('product_id'))

In [0]:
df.write.mode('overwrite').parquet('/FileStore/tables/transactions/transactions.parquet')